In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from bs4 import BeautifulSoup
import json
import matplotlib.pyplot as plt

In [ ]:
with open('../data/raw/ctg-studies.json') as f:
    data = json.load(f)

In [ ]:
import json
import csv

# Open JSON file
with open('../data/raw/ctg-studies.json') as f:
    data = json.load(f)

# Open CSV file in write mode
with open('../data/interim/ctg-studies.csv', 'w', newline='') as csvfile:
    # Create a CSV writer object
    writer = csv.writer(csvfile)

    # Write the header
    writer.writerow(['Key', 'Value'])  

    # Iterate through JSON data and write to CSV
    for item in data:
        for key, value in item.items():
            writer.writerow([key, value])



In [ ]:
data =pd.read_csv("../data/interim/ctg-studies.csv")

In [ ]:
data = data[(data["Key"] == "protocolSection") | (data["Key"] == "resultsSection")]

In [ ]:
data[data["Key"] == "protocolSection"].Value.unique()

In [ ]:
df = data.copy(deep=True)

In [ ]:
import ast
import spacy

In [ ]:
nlp = spacy.load("en_core_sci_sm")

In [ ]:
nct_ids = []
results = []

# Iterate through the DataFrame
for index, row in df.iterrows():
    # Check if the row represents protocolSection or resultsSection
    if row['Key'] == 'protocolSection':
        # Convert string representation of dictionary to actual dictionary
        value_dict = ast.literal_eval(row['Value'])
        # Extract nctId from the value
        nct_id = value_dict['identificationModule']['nctId']
        nct_ids.append(nct_id)
    elif row['Key'] == 'resultsSection':
        # Convert string representation of dictionary to actual dictionary
        value_dict = ast.literal_eval(row['Value'])
        # Extract results from the value
        results_data = value_dict
        results.append(results_data)

# Create a new DataFrame with extracted data
new_df = pd.DataFrame({'nctId': nct_ids, 'Results': results})

In [ ]:
def extract_info(row):
    entry = row['Results']
    result = {}

    # Extract 'started' and 'not_completed' values from participantFlowModule
    periods = entry['participantFlowModule'].get('periods', [])
    for period in periods:
        milestone_type = period.get('type', '')
        if milestone_type == 'STARTED':
            result['started'] = period['achievements'][0]['numSubjects']
        elif milestone_type == 'NOT COMPLETED':
            result['not_completed'] = period['achievements'][0]['numSubjects']
    
    return result

# Apply the function to every row of the DataFrame and convert to DataFrame
df_extracted = new_df.apply(extract_info, axis=1).apply(pd.Series)

# Concatenate the extracted data with the original DataFrame
df_concatenated = pd.concat([new_df, df_extracted], axis=1)

print(df_concatenated)


In [ ]:
def extract_info(row):
    entry = row['Results']
    started = None
    not_completed = None

    # Extract 'started' and 'not_completed' values from participantFlowModule
    periods = entry.get('participantFlowModule', {}).get('periods', [])
    for period in periods:
        print (period)
        milestone_type = period.get('type', '')
        if milestone_type == 'STARTED':
            for achievement in period.get('achievements', []):
                if achievement['groupId'] == 'FG000':
                    started = achievement['numSubjects']
        elif milestone_type == 'NOT COMPLETED':
            for achievement in period.get('achievements', []):
                if achievement['groupId'] == 'FG000':
                    not_completed = achievement['numSubjects']



In [ ]:
df=new_df.copy(deep=True)

In [ ]:
# Function to extract information from each row
def extract_info(row):
    entry = row['Results']
    # Initialize variables for FG000 and FG001
    started_fg000 = None
    not_completed_fg000 = None
    started_fg001 = None
    not_completed_fg001 = None

    # Initialize variables for FG002 to FG012
    fg_start = 2  # FG002 starts at index 2
    fg_end = 13  # FG012 ends at index 13
    for i in range(fg_start, fg_end + 1):
        globals()[f'started_fg{i:03d}'] = None
        globals()[f'not_completed_fg{i:03d}'] = None

    # Extract 'started' and 'not_completed' values from participantFlowModule
    periods = entry.get('participantFlowModule', {}).get('periods', [])
    for period in periods:
        for milestone in period.get('milestones', []):
            milestone_type = milestone.get('type', '')
            achievements = milestone.get('achievements', [])
            for achievement in achievements:
                group_id = achievement.get('groupId', '')
                num_subjects = achievement.get('numSubjects', '')
                # Check each group and assign values accordingly
                if group_id == 'FG000':
                    if milestone_type == 'STARTED':
                        started_fg000 = num_subjects
                    elif milestone_type == 'COMPLETED':
                        not_completed_fg000 = num_subjects
                elif group_id == 'FG001':
                    if milestone_type == 'STARTED':
                        started_fg001 = num_subjects
                    elif milestone_type == 'COMPLETED':
                        not_completed_fg001 = num_subjects
                elif group_id.startswith('FG'):
                    group_index = int(group_id[2:])
                    if fg_start <= group_index <= fg_end:
                        if milestone_type == 'STARTED':
                            globals()[f'started_fg{group_index:03d}'] = num_subjects
                        elif milestone_type == 'COMPLETED':
                            globals()[f'not_completed_fg{group_index:03d}'] = num_subjects

    # Return extracted data as a pandas Series
    return pd.Series({
        'started_fg000': started_fg000,
        'not_completed_fg000': not_completed_fg000,
        'started_fg001': started_fg001,
        'not_completed_fg001': not_completed_fg001,
        **{f'started_fg{i:03d}': globals()[f'started_fg{i:03d}'] for i in range(fg_start, fg_end + 1)},
        **{f'not_completed_fg{i:03d}': globals()[f'not_completed_fg{i:03d}'] for i in range(fg_start, fg_end + 1)}
    })

# Apply the function to every row of the DataFrame and convert to DataFrame
df_extracted = df.apply(extract_info, axis=1)

# Concatenate the extracted data with the original DataFrame
df_concatenated = pd.concat([df, df_extracted], axis=1)

print(df_concatenated)


In [ ]:
# Sum the values across columns for total_started
df_concatenated["total_started"] = df_concatenated.filter(like="started_fg").astype(float).sum(axis=1)

# Sum the values across columns for not_completed_total
df_concatenated["not_completed_total"] = df_concatenated.filter(like="not_completed_fg").astype(float).sum(axis=1)

#Calculate the percentage of the withdrawal rate
df_concatenated["percentage"] = df_concatenated["not_completed_total"]/ df_concatenated["total_started"]

In [ ]:
# Function to calculate the number of arms per row
def calculate_number_arms(row):
    for column in df_concatenated.filter(like="started_fg").columns:
        if pd.isna(row[column]):
            # Extract the number of arms from the column name
            return int(column.split("_")[1][3:])  # Extract the number after "FG00"
    return None  # If no NaN value found, return None

# Apply the function to calculate the number of arms per row
df_concatenated["number_arms"] = df_concatenated.apply(calculate_number_arms, axis=1)


In [ ]:
df_numbers = df_concatenated.copy(deep=True)

In [ ]:
df_numbers_keep = df_numbers[["total_started","nctId"]]

In [ ]:

# Function to extract group information
def extract_group_info(row):
    entry = row['Results']
    groups = entry.get('participantFlowModule', {}).get('groups', [])
    group_info = {}

    for group in groups:
        group_id = group.get('id', '')
        title = group.get('title', '')
        group_info[group_id] = title

    # Initialize a dictionary to store information for FG000 to FG013
    fg_info = {'FG{:03d}'.format(i): None for i in range(14)} 

    for group_id, title in group_info.items():
        fg_info[group_id] = title

    return pd.Series(fg_info)

# Apply the function to every row of the DataFrame and convert to DataFrame
df_group_info = df.apply(extract_group_info, axis=1)

# Concatenate the extracted group information with the original DataFrame
df_with_group_info = pd.concat([df, df_group_info], axis=1)



In [ ]:
df_with_group_info = df_with_group_info.astype(str)

In [ ]:
def process_text(text):
    doc = nlp(text)
    # Perform tokenization and entity recognition
    interventions = []
    for ent in doc.ents:
        if ent.label_ == "DRUG":  # Example: Identify drug entities
            interventions.append(ent.text)
    return interventions


In [ ]:
df_with_group_info['interventions'] = df_with_group_info['FG000'].apply(process_text)

In [ ]:
df_numbers_combine = df_numbers.copy(deep=True)

In [ ]:
df_numbers_combine = df_numbers_combine[["nctId","total_started","not_completed_total","percentage","number_arms"]]

In [ ]:
df_NER_design = df_with_group_info[["nctId","FG000","FG001"]]

In [ ]:
df_with_group_info.Results.tolist()

In [ ]:
import pandas as pd

# Function to extract group information
def extract_group_info(row):
    entry = row['Results']
    groups = entry.get('participantFlowModule', {}).get('groups', [])
    group_info = {}

    for group in groups:
        group_id = group.get('id', '')
        title = group.get('title', '')
        group_info[group_id] = title

    # Initialize a dictionary to store information for FG000 to FG013
    fg_info = {'FG{:03d}'.format(i): None for i in range(14)} 

    for group_id, title in group_info.items():
        fg_info[group_id] = title

    return pd.Series(fg_info)

# Apply the function to every row of the DataFrame and convert to DataFrame
df_groups = df.apply(extract_group_info, axis=1)

# Concatenate the extracted group information with the original DataFrame
df_groups = pd.concat([df, df_groups], axis=1)

print(df_with_group_info)


In [ ]:
def extract_drop_withdraws(row):
    # Access the 'participantFlowModule' key
    participant_flow_module = row['Results'].get('participantFlowModule', {})
    
    # Get the 'periods' list from the participantFlowModule dictionary
    periods = participant_flow_module.get('periods', [])
    
    # Initialize an empty dictionary to store reasons
    reasons_dict = {}
    
    # Iterate through each period in the 'periods' list
    for period in periods:
        # Get the 'dropWithdraws' list from each period
        drop_withdraws = period.get('dropWithdraws', [])
        
        # Iterate through each withdrawal in the 'dropWithdraws' list
        for withdrawal in drop_withdraws:
            # Get the withdrawal type
            reason_type = withdrawal.get('type', '')  
            
            # Iterate through each reason in the 'reasons' list within the withdrawal
            for reason in withdrawal.get('reasons', []):
                # Get the group ID and number of subjects
                reason_group = reason['groupId']
                num_subjects = int(reason['numSubjects'])
                
                # Construct a key using the type and group ID
                reason_key = f'{reason_type}_{reason_group}'
                
                # Update the dictionary with the number of subjects for each reason
                if reason_key in reasons_dict:
                    reasons_dict[reason_type] += num_subjects
                else:
                    reasons_dict[reason_type] = num_subjects
    
    return pd.Series(reasons_dict)


df_reasonswithdraw= df.apply(extract_drop_withdraws, axis=1)

# Concatenate the extracted group information with the original DataFrame
df_reasonswithdraw = pd.concat([df, df_reasonswithdraw], axis=1)

print(df_reasonswithdraw)



In [ ]:
df_reasonswithdraw.drop(columns=["Results"],inplace=True)

In [ ]:
df_reasonswithdraw = df_reasonswithdraw.set_index(["nctId"])

In [ ]:
df_plot = pd.merge(df_reasonswithdraw.reset_index(), df_numbers_keep, on=["nctId"])

In [ ]:
# Divide each value by the corresponding total_started value
normalized_reason_counts = df_plot.iloc[:, 1:704].div(df_plot["total_started"], axis=0)
# Sum the values for each column (vertically)
column_sums = normalized_reason_counts.sum(axis=0)


In [ ]:
# Sum the values along each column, selecting the desired columns

# Find the column with the maximum sum
most_entries_reason = column_sums.idxmax()

# Get the count of entries for the most common reason
most_entries_count = column_sums.max()

print("Reason with the most withdraws:", most_entries_reason)
print("Number of withdraws:", most_entries_count)


In [ ]:
from wordcloud import WordCloud
# Sum the values along each column and sort in descending order
reason_counts = normalized_reason_counts.sum(axis=0).sort_values(ascending=False)

# Take the top 20 reasons
top_20_reasons = reason_counts.head(40)

# Create a WordCloud object
wordcloud = WordCloud(width=1200, height=800, background_color='white').generate_from_frequencies(top_20_reasons)

# Display the generated word cloud
plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('')
plt.savefig("Wordcloudnormalised.pdf")




In [ ]:
palette = sns.color_palette()
first_color = palette[0]

In [ ]:
import matplotlib.pyplot as plt

# Sum the values along each column and sort in descending order
reason_counts = df_reasonswithdraw.sum(axis=0).sort_values(ascending=False)

# Take the top 20 reasons
top_20_reasons = reason_counts.head(20)

# Create a bar plot
plt.figure(figsize=(10, 6))
top_20_reasons.plot(kind='bar', color=first_color)
plt.title('Top 20 Reasons with the Most Withdrawals')
plt.xlabel('Reason')
plt.ylabel('Number of Entries')

# Add letters as labels for each reason
for i, (reason, count) in enumerate(top_20_reasons.items()):
    plt.text(i, count, chr(ord('A') + i), ha='center', va='bottom')

plt.tight_layout()
plt.show()


In [ ]:
# Assuming df is your DataFrame
total_sum = df_reasonswithdraw.sum().sum()
print("Total participants that withdrew:", total_sum)


In [ ]:
df.apply(extract_drop_withdraws, axis=1)

In [ ]:
df.apply(extract_drop_withdraws, axis=1).columns.tolist()

In [ ]:
import pandas as pd

# Function to extract drop/withdraw reasons and numSubjects -- DID NOT WORK 
def extract_drop_withdraws(row):
    entry = row['Results']
    withdrawals = entry.get('participantFlowModule', {}).get('dropWithdraws', [])
    reasons_dict = {}
    for withdrawal in withdrawals:
        reason_type = withdrawal.get('type', '')  # Access 'type' within each withdrawal
        for reason in withdrawal.get('reasons', []):
            reason_group = reason['groupId']
            num_subjects = int(reason['numSubjects'])
            reason_key = f'{reason_type}_{reason_group}'
            if reason_key in reasons_dict:
                reasons_dict[reason_key] += num_subjects
            else:
                reasons_dict[reason_key] = num_subjects
    return pd.Series(reasons_dict)

# Apply the function to every row of the DataFrame and convert to DataFrame
df_with_withdraw_info = df.apply(extract_drop_withdraws, axis=1)

# Concatenate the extracted information with the original DataFrame
df_with_withdraw_info = pd.concat([df, df_with_withdraw_info], axis=1)

print(df_with_withdraw_info)


In [ ]:
import pandas as pd

# Function to extract group information
def extract_group_info(row):
    entry = row['Results']
    groups = entry.get('participantFlowModule', {}).get('groups', [])
    group_info = {}

    for group in groups:
        group_id = group.get('id', '')
        title = group.get('title', '')
        if group_id.startswith('FG'):
            group_info[group_id] = title

    # Initialize group titles from FG000 to FG012 with None
    fg_titles = {f'FG{i:03d}': None for i in range(13)}

    # Update titles for existing groups
    fg_titles.update(group_info)

    return pd.Series(fg_titles)

# Apply the function to every row of the DataFrame and convert to DataFrame
df_group_info = df.apply(extract_group_info, axis=1)

# Concatenate the extracted group information with the original DataFrame
df_with_group_info = pd.concat([df, df_group_info], axis=1)

print(df_with_group_info)


In [ ]:
df_inter= pd.merge(df_with_group_info, df_numbers_combine, on="nctId")

In [ ]:
import pandas as pd

# Function to extract baseline characteristics information
def extract_baseline_info(row):
    entry = row['Results']
    measures = entry.get('baselineCharacteristicsModule', {}).get('measures', [])
    
    # Initialize variables to store extracted information
    mean_age = None
    std_age = None
    num_female = None
    num_male = None

    # Iterate over measures to extract required information
    for measure in measures:
        title = measure.get('title', '')
        if title == 'Age, Continuous':
            for class_info in measure.get('classes', []):
                for category_info in class_info.get('categories', []):
                    measurements = category_info.get('measurements', [])
                    for measurement in measurements:
                        group_id = measurement.get('groupId', '')
                        value = measurement.get('value', '')
                        spread = measurement.get('spread', '')
                        if group_id == 'BG000':
                            mean_age = value
                            std_age = spread
                        elif group_id == 'BG001':
                            mean_age = value
                            std_age = spread
                        elif group_id == 'BG002':
                            mean_age = value
                            std_age = spread
        elif title == 'Sex: Female, Male':
            for class_info in measure.get('classes', []):
                for category_info in class_info.get('categories', []):
                    title = category_info.get('title', '')
                    measurements = category_info.get('measurements', [])
                    for measurement in measurements:
                        group_id = measurement.get('groupId', '')
                        value = measurement.get('value', '')
                        if title == 'Female':
                            if group_id == 'BG000':
                                num_female = value
                            elif group_id == 'BG001':
                                num_female = value
                            elif group_id == 'BG002':
                                num_female = value
                        elif title == 'Male':
                            if group_id == 'BG000':
                                num_male = value
                            elif group_id == 'BG001':
                                num_male = value
                            elif group_id == 'BG002':
                                num_male = value
    
    # Return extracted information as a pandas Series
    return pd.Series({
        'mean_age': mean_age,
        'std_age': std_age,
        'num_female': num_female,
        'num_male': num_male
    })

# Apply the function to every row of the DataFrame and convert to DataFrame
df_baseline_info = df.apply(extract_baseline_info, axis=1)

# Concatenate the extracted baseline information with the original DataFrame
df_with_baseline_info = pd.concat([df, df_baseline_info], axis=1)

print(df_with_baseline_info)


In [ ]:
df_numbers_combine = df_numbers[["nctId","number_arms", "percentage", "total_started", "not_completed_total"]]

In [ ]:
df_merged = pd.merge(df_numbers_combine,df_with_baseline_info, on="nctId")

In [ ]:
df_merged = df_merged.set_index(["nctId"])

In [ ]:
df_merged = df_merged.drop(columns=["Results"])

In [ ]:
df_merged ["percentage"] = df_merged["percentage"].astype(float)

In [ ]:
df_merged.replace({None: np.nan}, inplace=True)
df_merged = df_merged.replace("",np.nan)
df_merged = df_merged.replace("NA",np.nan)

In [ ]:
df_merged = df_merged.astype(float)

# Plotting

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Sum the values along each column and sort in descending order
interventions_counts = df_inter.groupby(["FG000"])["percentage"].mean().sort_values(ascending=False)

# Take the top 20 reasons
top_20_reasons = interventions_counts.head(40)

# Create a WordCloud object
wordcloud_int= WordCloud(width=1200, height=800, background_color='white').generate_from_frequencies(top_20_reasons)

# Display the generated word cloud
plt.figure(figsize=(10, 6))
plt.imshow(wordcloud_int, interpolation='bilinear')
plt.axis('off')
plt.title('')

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.cm import viridis


color = viridis(0.5)  
plt.figure(figsize=(10, 6))
plt.scatter(df_merged['mean_age'], df_merged['percentage'], color=color, alpha=0.5)
plt.title('Retention Percentage vs. Average Cohort Age')
plt.xlabel('Average Age')
plt.ylabel('Retention Percentage')
plt.grid(True)
plt.show()


In [ ]:

color = viridis(0.3) 

# Filter the DataFrame based on the condition
filtered_df = df_merged[df_merged['std_age'] < 50]

plt.figure(figsize=(10, 6))
plt.scatter(filtered_df['std_age'], filtered_df['percentage'], color=color, alpha=0.5)
plt.title('Retention Percentage vs. Standard Deviation Cohort Age')
plt.xlabel('Standard Deviation Age')
plt.ylabel('Retention Percentage')
plt.grid(True)
plt.show()



In [ ]:
df_merged["percentage_female"] = df_merged["num_female"].div(df_merged["total_started"])
df_merged["percentage_male"] = df_merged["num_male"].div(df_merged["total_started"])

In [ ]:
# Select a color from Viridis colormap
color = viridis(0.1)  
plt.figure(figsize=(10, 6))
plt.scatter(df_merged[df_merged["percentage_female"] <= 1]['percentage_female'], df_merged[df_merged["percentage_female"] <= 1]['percentage'], color=color, alpha=0.5)
plt.title('Retention Percentage vs. Percentage of Female participants')
plt.xlabel('Percentage of Female participants')
plt.ylabel('Retention Percentage')
plt.grid(True)
plt.show()


In [ ]:
color = viridis(0.7)  # Adjust the value between 0 and 1 to select the desired color

plt.figure(figsize=(10, 6))
plt.scatter(df_merged[df_merged["percentage_male"] <= 1]['percentage_male'], df_merged[df_merged["percentage_male"] <= 1]['percentage'], color=color, alpha=0.5)
plt.title('Retention Percentage vs. Percentage of Male participants')
plt.xlabel('Percentage of Male participants')
plt.ylabel('Retention Percentage')
plt.grid(True)
plt.show()


In [ ]:


arms_counts = df_merged['number_arms'].value_counts()

arms_counts_df = pd.DataFrame(arms_counts).reset_index()

arms_counts_df.columns = ['number_arms', 'number_of_studies']



In [ ]:
df_merged_arms = pd.merge(arms_counts_df, df_merged, on="number_arms")

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.cm import viridis


mean_retention_by_arms = df_merged_arms.groupby('number_arms')['percentage'].mean()


color = viridis(0.3)  


plt.figure(figsize=(10, 6))
plt.plot(mean_retention_by_arms.index, mean_retention_by_arms.values, marker='o', linestyle='-', color=color)
plt.title('Average Retention Percentage by Number of Arms')
plt.xlabel('Number of Arms')
plt.ylabel('Average Retention Percentage')
plt.grid(True)
plt.xticks(mean_retention_by_arms.index)  
plt.show()


In [ ]:
color = viridis(0.3)  


plt.figure(figsize=(10, 6))
plt.scatter(df_merged_arms[df_merged_arms["percentage_male"] <= 1]['percentage_male'], df_merged[df_merged["percentage_male"] <= 1]['percentage'], color=color, alpha=0.5)
plt.title('Retention Percentage vs. Percentage of Male participants')
plt.xlabel('Percentage of Male participants')
plt.ylabel('Retention Percentage')
plt.grid(True)
plt.show()


In [ ]:
X = df_merged.drop(columns=['percentage',"not_completed_total"])
X.fillna(0, inplace=True) 

X = X.astype(float)

y = df_merged['percentage']
y= y.fillna(0)

# Simple regression models


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_regressor = RandomForestRegressor(n_estimators=100, random_state=42)


rf_regressor.fit(X_train, y_train)


y_pred = rf_regressor.predict(X_test)


mae = mean_absolute_error(y_test, y_pred)
print("Mean absolut Error:", mae)

In [ ]:


# Get feature importances from the trained model
feature_importances = rf_regressor.feature_importances_

# Get the names of the features
feature_names = X.columns

# Sort feature importances in descending order
sorted_indices = feature_importances.argsort()[::-1]
sorted_feature_importances = feature_importances[sorted_indices]
sorted_feature_names = feature_names[sorted_indices]

# Plot the feature importances
plt.figure(figsize=(10, 6))
plt.bar(range(len(sorted_feature_importances)), sorted_feature_importances, tick_label=sorted_feature_names)
plt.xlabel('Feature')
plt.ylabel('Importance')
plt.title('Feature Importances')
plt.xticks(rotation=90)
plt.show()


In [ ]:
json_data = new_df["Results"]
df_objects = pd.json_normalize(json_data)

# Trying pretrained LLM 


In [ ]:
from transformers import BertTokenizer, BertForTokenClassification

In [ ]:
import tensorflow

In [ ]:
tensorflow.__version__

In [ ]:
df_NER_design = df_NER_design.astype(str)

In [ ]:
def perform_ner(text):
    doc = nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    return entities

# Apply NER to specified columns
for column in ["FG000", "FG001"]:
    df_NER_design[column + "_ner"] = df_NER_design[column].apply(perform_ner)

# Display the DataFrame with NER results
print(df_NER_design)

In [ ]:
df_NER_design.FG000.unique().tolist()

In [ ]:
nctids = new_df.nctId.unique().tolist()

In [ ]:
file_path = '../data/processed/nctids.txt'

# Open the file in write mode
with open(file_path, 'w') as file:
    # Write each nctId to the file
    for nctid in nctids:
        file.write(nctid + '\n')


In [ ]:
new_df.to_csv("../data/processed/results.csv")

In [ ]:
# Create a function to extract information from each dictionary entry and return it as a dictionary
def extract_info(data_dict):
    result = {}
    for entry in data_dict:
        participant_flow = data_dict[entry].get('participantFlowModule', {})
        groups = participant_flow.get('groups', [{}])
        milestones = groups[0].get('milestones', [{}])
        achievements = milestones[0].get('achievements', [{}])
        
        result[f'number_of_participants_started_{entry}'] = achievements[0].get('numSubjects', '')
        
        drop_withdraws = participant_flow.get('dropWithdraws', [{}])
        reasons = drop_withdraws[0].get('reasons', [{}])
        
        result[f'number_withdrawn_{entry}'] = reasons[0].get('numSubjects', '')
        result[f'description_{entry}'] = data_dict[entry].get('descriptionModule', {}).get('briefSummary', '')
    return result

# Apply the function to the 'Results' column and expand the result into separate columns
expanded_data = new_df['Results'].apply(extract_info).apply(pd.Series)

# Concatenate the expanded data with the original DataFrame
df_expanded = pd.concat([new_df, expanded_data], axis=1)


In [ ]:
test = new_df.iloc[25:30,:]

In [ ]:
targets = [26,31,20,0,0,40,3,22,8,36]
filtered_df["target"] = targets

In [ ]:
filtered_df.to_csv("../data/processed/train.csv")

In [ ]:
dataset = load_dataset("csv", data_files="../data/processed/train.csv")

split_ratio = 0.8
num_samples = len(dataset['train'])
num_train = int(split_ratio * num_samples)
num_test = num_samples - num_train
train_indices = list(range(num_train))
test_indices = list(range(num_train, num_samples))
train_data = {key: [dataset['train'][key][i] for i in train_indices] for key in dataset['train'].column_names}
test_data = {key: [dataset['train'][key][i] for i in test_indices] for key in dataset['train'].column_names}
split_dataset = DatasetDict({'train': Dataset.from_dict(train_data), 'test': Dataset.from_dict(test_data)})

In [ ]:
def preprocess_function(examples):
    label = examples["target"] 
    examples = tokenizer(examples["Results"], truncation=True, padding="max_length", max_length=256)
    examples["target"] = label
    return examples


In [ ]:
tokenized_data= split_dataset.map(preprocess_function, batched=True)

In [ ]:
tokenized_data['train'][0]
data_collactor = DataCollatorWithPadding(tokenizer=tokenizer)


In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

def compute_metrics_for_regression(eval_pred):
    logits, labels = eval_pred 
    labels = labels.reshape(-1, 1)
    
    mse = mean_squared_error(labels, logits)
    mae = mean_absolute_error(labels, logits)
    r2 = r2_score(labels, logits)
    single_squared_errors = ((logits - labels).flatten()**2).tolist()
    accuracy = sum([1 for e in single_squared_errors if e < 0.25]) / len(single_squared_errors)
    return {"mse": mse, "mae": mae, "r2": r2, "accuracy": accuracy}
    

In [ ]:
BASE_MODEL = "medicalai/ClinicalBERT"
LEARNING_RATE = 2e-5
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 20


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="../outputs/model",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    metric_for_best_model="accuracy",
    load_best_model_at_end=True,
    weight_decay=0.01,
)

In [ ]:
input = {
    "input_ids": tokenized_data["train"]["input_ids"],
    "attention_mask": tokenized_data["train"]["attention_mask"],
}

target = tokenized_data["train"]["target"]
target_tensor = torch.tensor(target, dtype=torch.float32)


In [ ]:
class RegressionTrainer(Trainer):
    def compute_loss(self, model, input, return_outputs=False):
        labels = target_tensor  # Accessing labels using the key "target"
        outputs = model(**input)
        logits = outputs[0][:, 0]
        loss = torch.nn.functional.mse_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss
